# 27. Text-Image FN-Masked — Unfrozen-Layer Selection (last `n` layers)

Aligned with **notebook 28** (`28.textimage_inbatch_remaining_sizes`) and **notebook 29** (`29.textimage_inbatch_mining_remaining_sizes`) on logging, HF cache, results layout, pickled-variable strategy, chunked encoding, and train-or-resume semantics.

**What this experiment does differently:**

| Axis | NB28 / NB29 | NB27 (this notebook) |
|---|---|---|
| **Train size** | Variable (per size) | **Fixed at 2500** |
| **Epochs** | 20 | **20** |
| **Variable swept** | Data size (`size_{N}`) | **Number of unfrozen layers** (`last_{n}_layers`) |
| **Loss** | NB28: FN masks; NB29: in-batch mining | **NB28's FN-masking loss** (`masked_symmetric_infonce`) |
| **Output dir per run** | `size_{N}/` | `last_{n}_layers/` |
| **Per-epoch GT** | Logged only (NB28/29 post-alignment) | **Logged + saved every 2 epochs** to `per_epoch_gt/epoch_NN_gt_results.csv` — 10 CSVs per run (epochs 2, 4, …, 20) |

### `n = 0` — the fully-frozen baseline

- Both encoders fully frozen; only projection heads train.
- Folder `last_0_layers/` uses the same `last_{n}_layers` regex as every other `n`, so Section 10 roll-ups work uniformly.
- **Every improvement above `n=0` is attributable to unfreezing**, not to the projection heads learning.
- Optimizer for `n=0` contains only the head param-group (encoder groups are omitted, not passed as empty lists — see Section 4).
- Sanity check's gradient test is `n`-aware: at `n=0` it logs "expected absent", at `n>0` it **raises** if the encoders aren't receiving gradients.


### Output layout

```
OUTPUT_PARENT = results/runs/section_4/diff_unfrozen_layers/textimage_fn_masked
  img_inbatch_ft.log
  summary_all_n.csv                         # final GT per n, plus NB28 n=6 reference
  summary_per_epoch_all_n.csv               # per-(n, epoch, direction) rows for NB27's runs
  last_0_layers/                            # frozen baseline
    latest_checkpoint.pt
    training_history.csv                    # columns: epoch, imagetext_loss (same as NB28)
    gt_results.csv
    per_epoch_gt/epoch_02,04,06,...,20_gt_results.csv
    head_text_bge.pt
    head_image_dinov3.pt
    dinov3_model_unfrozen.pt
    bge_model_unfrozen.pt
  last_2_layers/  last_4_layers/  last_6_layers/  last_8_layers/
```

Artefact names inside each `last_{n}_layers/` are **byte-for-byte identical** to NB28 / NB29's `size_{N}/` layout (`_MODEL_FILE_RENAMES`).

### GT cadence — every 2 epochs

`GT_EVERY_N_EPOCHS = 2` → GT is evaluated at epochs **2, 4, 6, …, 20** (10 evaluations per run). `per_epoch_gt/` contains exactly those CSVs (epochs 1, 3, 5, … are training-only and produce no CSV). Final `gt_results.csv` is always the epoch-20 result. The last epoch is always evaluated regardless of `GT_EVERY_N_EPOCHS` (guarded in code).

### Controlled-comparison design

All `n` values see the **same 2500 records** (`random.Random(RANDOM_SEED + SIZE)`), and each run reseeds `np.random.seed(RANDOM_SEED)` before its epoch loop, so batch order is **identical across `n`** — any difference in the results is attributable to layer count, not sampling noise.

### Requires NB28's pickles

This notebook **loads** NB28's pickles (`vars/raw_textimage.pkl`, `vars/nb28_textimage_records.pkl` [33526 records, with `section_title`], `vars/raw_gt_textimage.pkl`, `vars/gt_textimage_records.pkl`). If missing, run NB28 Section 2 + Section 6 once first.

### Train-or-resume semantics (matches NB28 / NB29)

| State on disk (per `n`) | Behaviour |
|---|---|
| `last_{n}_layers/gt_results.csv` **exists** | Skip; return existing summary |
| `last_{n}_layers/latest_checkpoint.pt` **exists**, epoch < 20 | Resume |
| `last_{n}_layers/latest_checkpoint.pt` **exists**, epoch ≥ 20 | Finalize only |
| `last_{n}_layers/latest_checkpoint.pt` **missing** | **Train from scratch**, epoch 0 → 20 |

In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 0 — Installations

Same pin set as NB28 / NB29.

In [1]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6   # downgrade for DINOv3
# !pip install --upgrade huggingface_hub

### dinov3 is a gated model — ask for grant on HuggingFace, then log in once before the first download.

In [2]:
# from huggingface_hub import login
# login()

## Section 1 — Setup, logging, config

Logging, HF cache, and pickle helpers are **identical** to NB28 / NB29 — one shared log file rooted at this experiment's parent, explicit `HF_CACHE` on every `from_pretrained`, `pkl_vars` / `reload_vars`.

In [1]:
import json
import os
import re
import sys
import time
import math
import random
import pickle
import logging
import gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

import transformers

In [2]:
transformers.logging.set_verbosity_error()

# --- Output root for THIS experiment (all n share one log) ---
OUTPUT_PARENT = str(RESULTS_RUNS_ROOT / 'layer_sweeps' / 'textimage_fn_masked')
os.makedirs(OUTPUT_PARENT, exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_PARENT, "img_inbatch_layers_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: %s", device)
if device == "cuda":
    logger.info("GPU: %s", torch.cuda.get_device_name(0))


# --- Pickle helpers (identical to NB28 / NB29) ---
VARS_DIR = str(FINETUNING_ROOT / "vars")
os.makedirs(VARS_DIR, exist_ok=True)

def pkl_vars(varname, filename):
    with open(filename, 'wb') as file:
        pickle.dump(varname, file)

def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

2026-10-21 11:21:38,199 : INFO : Device: cuda
2026-10-21 11:21:38,202 : INFO : GPU: Quadro RTX 8000


### Config

- **`SIZE = 2500`** — fixed training-set size for every `n`.
- **`TARGET_EPOCHS = 20`** — every run trained to 20 epochs.
- **`GT_EVERY_N_EPOCHS = 2`** — GT evaluated & saved at epochs 2, 4, 6, …, 20. Last epoch always evaluated (guarded).
- **`N_LAYERS_TO_TRY = [0, 2, 4, 6, 8]`** — the swept axis; `n=6` is trained in this notebook.
- **`NB28_N6_REFERENCE_GT_PATH`** — optional comparison row from NB28's `size_2500/gt_results.csv`; NB28 uses a 512-token BGE limit, so this reference is not context-matched to NB27.
- **`BGE_MAX_LENGTH = 8192`** — NB27-specific long-context setting; NB28/29 text-image size-scaling runs use 512 tokens.
- **`GT_EVAL_CHUNK_SIZE = 16`** — GT examples are batched in groups of 16 for memory; each text is still encoded with the 8192-token limit.
- **`IMAGE_ENCODE_CHUNK_SIZE = 16`** — batched + chunked DINOv3 image encoding.
- **`_MODEL_FILE_RENAMES`** — same dict as NB28, for on-disk filename parity.

In [3]:
# --- Shared (aligned with NB28) ---
HF_CACHE          = DEFAULT_HF_CACHE
RANDOM_SEED       = 42
SIZE              = 2500                   # fixed training-set size for every n
TARGET_EPOCHS     = 20
GT_EVERY_N_EPOCHS = 2                      # GT at epochs 2,4,6,...,20
N_LAYERS_TO_TRY   = [0, 2, 4, 6, 8]           # 0 = fully frozen baseline; n=6 is trained here

# n=6 reference: NB28's size_2500 run (FN masking, n=6, same seed, same 2500 subset)
NB28_N6_REFERENCE_GT_PATH = str(
    RESULTS_RUNS_ROOT / "size_sweeps" / "TextImage_Smaller_train_sizes_runs"
    / "size_2500" / "gt_results.csv"
)

# --- Chunking ---
IMAGE_ENCODE_CHUNK_SIZE = 16               # DINOv3 image encoding (train + GT eval)
GT_EVAL_CHUNK_SIZE      = 16               # batch GT text examples (max_len=8192)

LOG_EVERY_N_STEPS        = 50
CHECKPOINT_EVERY_N_STEPS = 100
MAX_RUNTIME_HOURS_PER_SIZE = 10.0

K_VALUES          = [1, 3, 5, 10]
HEAD_LEARNING_RATE = 1e-4
BGE_ENCODER_LR    = 1e-5
DINOV3_ENCODER_LR = 1e-6
TEMPERATURE       = 0.07
SHARED_DIM        = 512
BATCH_SIZE        = 32
BGE_MODEL_NAME    = "BAAI/bge-m3"
BGE_MAX_LENGTH    = 8192                   # NB27 layer sweep; NB28/29 scaling uses 512
DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"
FN_SIMILARITY_THRESHOLD = 0.8

_MODEL_FILE_RENAMES = {
    "dinov3_model": "dinov3_model_unfrozen",
    "bge_model":    "bge_model_unfrozen",
}

# --- INPUT paths (unchanged) ---
FN_METADATA_PATH    = str(FINETUNING_ROOT / 'Shared_data' / 'df_combined_metadata.pkl')
FN_IMAGE_IMAGE_PATH = str(FINETUNING_ROOT / 'Shared_data' / 'reranked_image_image.pkl')
IMG_BASE_DIR        = str(BSV_ROOT / 'Automatic Construction')
TEXTIMAGE_PAIRS_PATH = os.path.join(IMG_BASE_DIR, "bsv_pairs_auto_textimage.jsonl")
IMAGE_DIR           = os.path.join(IMG_BASE_DIR, "bsv_images_auto_textimage")
GT_TEXTIMAGE_PATH   = str(BSV_ROOT / 'Ground Truth' / 'bsv_gt_textimage_final_500.jsonl')
GT_IMAGE_DIR        = str(BSV_ROOT / 'Ground Truth' / 'bsv_images_gt_2404')

logger.info("Config loaded. OUTPUT_PARENT=%s", OUTPUT_PARENT)
logger.info("SIZE=%d, TARGET_EPOCHS=%d, GT_EVERY_N_EPOCHS=%d, N_LAYERS_TO_TRY=%s (0=frozen baseline; includes n=6)",
            SIZE, TARGET_EPOCHS, GT_EVERY_N_EPOCHS, N_LAYERS_TO_TRY)
logger.info("NB28 n=6 reference path: %s (exists=%s)",
            NB28_N6_REFERENCE_GT_PATH, os.path.exists(NB28_N6_REFERENCE_GT_PATH))
logger.info("Chunks: IMAGE_ENCODE_CHUNK_SIZE=%d, GT_EVAL_CHUNK_SIZE=%d, log_every=%d, ckpt_every=%d",
            IMAGE_ENCODE_CHUNK_SIZE, GT_EVAL_CHUNK_SIZE, LOG_EVERY_N_STEPS, CHECKPOINT_EVERY_N_STEPS)

2026-10-21 11:21:38,300 : INFO : Config loaded. OUTPUT_PARENT=/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_fn_masked
2026-10-21 11:21:38,302 : INFO : SIZE=2500, TARGET_EPOCHS=20, GT_EVERY_N_EPOCHS=2, N_LAYERS_TO_TRY=[0, 2, 4, 6, 8] (0=frozen baseline; 6 skipped -> NB28 ref)
2026-10-21 11:21:38,305 : INFO : NB28 n=6 reference path: /projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/Smaller_train_sizes_runs/size_2500/gt_results.csv (exists=True)
2026-10-21 11:21:38,306 : INFO : Chunks: IMAGE_ENCODE_CHUNK_SIZE=16, GT_EVAL_CHUNK_SIZE=16, log_every=50, ckpt_every=100


## Section 2 — Load data (reuse NB28's pickles)

All four pickles are **the exact files NB28 writes**:
- `vars/raw_textimage.pkl` — raw jsonl dump
- `vars/nb28_textimage_records.pkl` — **33526 records** with `{orig_idx, bulletin_id, section_title, text, image_path}` (needed by the FN masks)
- `vars/raw_gt_textimage.pkl` — raw GT jsonl
- `vars/gt_textimage_records.pkl` — `{gt_idx, text, image_path}`

If any of these are missing, run NB28's Section 2 + Section 6 once to build them. This notebook deliberately does **not** duplicate NB28's data-building chain.

In [4]:
REQUIRED_PICKLES = {
    "raw_textimage":        os.path.join(VARS_DIR, "raw_textimage.pkl"),
    "nb28_textimage_records": os.path.join(VARS_DIR, "nb28_textimage_records.pkl"),
    "raw_gt_textimage":     os.path.join(VARS_DIR, "raw_gt_textimage.pkl"),
    "gt_textimage_records": os.path.join(VARS_DIR, "gt_textimage_records.pkl"),
}
missing = [p for p in REQUIRED_PICKLES.values() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing NB28 pickles:\n  " + "\n  ".join(missing) +
        "\nRun NB28's Section 2 and Section 6 once to build them, then re-run this cell."
    )

raw_textimage        = reload_vars(REQUIRED_PICKLES["raw_textimage"])
textimage_records    = reload_vars(REQUIRED_PICKLES["nb28_textimage_records"])
raw_gt_textimage     = reload_vars(REQUIRED_PICKLES["raw_gt_textimage"])
gt_textimage_records = reload_vars(REQUIRED_PICKLES["gt_textimage_records"])

logger.info("Loaded NB28 pickles: %d training records, %d GT records",
            len(textimage_records), len(gt_textimage_records))
assert len(textimage_records) == 33526, f"expected 33526 records, got {len(textimage_records)}"
assert len(textimage_records) > SIZE, f"pool ({len(textimage_records)}) must exceed SIZE ({SIZE})"
assert len(gt_textimage_records) > 0

_sample = textimage_records[0]
for _k in ("bulletin_id", "section_title", "text", "image_path"):
    assert _k in _sample, f"missing key {_k!r} in textimage_records -- wrong pickle?"
logger.info("Training record schema OK (has bulletin_id/section_title/text/image_path)")

2026-10-21 11:21:38,947 : INFO : Loaded NB28 pickles: 33526 training records, 500 GT records
2026-10-21 11:21:38,949 : INFO : Training record schema OK (has bulletin_id/section_title/text/image_path)


## Section 3 — False-negative mask (image-image, verbatim from NB28)

Precomputed FN pairs at `colbert ≥ FN_SIMILARITY_THRESHOLD` are treated as false negatives and masked out of the in-batch contrastive loss. Keys are `(bulletin_id, section_title, text)`.

In [5]:
def build_key(bulletin_id, section_title, text):
    return (bulletin_id, section_title, text)

def build_fn_mask_set(pkl_path, threshold=FN_SIMILARITY_THRESHOLD):
    df = pd.read_pickle(pkl_path)
    hi = df[df["colbert"] >= threshold]
    pairs = set()
    for _, row in hi.iterrows():
        pairs.add(frozenset({int(row["global_i"]), int(row["global_j"])}))
    return pairs

fn_metadata = pd.read_pickle(FN_METADATA_PATH)

fn_image_image = build_fn_mask_set(FN_IMAGE_IMAGE_PATH)
key_to_globals_image = defaultdict(list)
for _, row in fn_metadata[fn_metadata["origin"] == "text_image"].iterrows():
    k = build_key(row["bulletin_id"], row["section_title"], row["text"])
    key_to_globals_image[k].append(row["_global"])
logger.info("Image-image: %d flagged pairs, %d unique keys", len(fn_image_image), len(key_to_globals_image))

def is_masked_image_image(record_i, record_j, threshold_set=fn_image_image):
    key_i = build_key(record_i["bulletin_id"], record_i["section_title"], record_i["text"])
    key_j = build_key(record_j["bulletin_id"], record_j["section_title"], record_j["text"])
    for gi in key_to_globals_image.get(key_i, []):
        for gj in key_to_globals_image.get(key_j, []):
            if frozenset({gi, gj}) in threshold_set:
                return True
    return False

logger.info("is_masked_image_image() ready.")

2026-10-21 11:21:45,240 : INFO : Image-image: 240959 flagged pairs, 27476 unique keys
2026-10-21 11:21:45,243 : INFO : is_masked_image_image() ready.


## Section 4 — Model builder (parameterized by `n_unfrozen_layers`)

Same architecture, LRs, and chunked image encoder as NB28. Only change: `n_unfrozen_layers` selects how many of the **last** layers to unfreeze in *both* encoders.

**`n = 0` handling:**
- Encoders fully frozen; only projection heads train.
- **Optimizer omits empty encoder param-groups** rather than passing `{"params": []}`. Head-only group is the single group for `n=0`.
- Checkpoint/resume round-trips cleanly because each `last_{n}_layers/` holds a checkpoint written with the group count that `n` produces.

Guards: `n < 0` → asserted away; `n > available` → warn and unfreeze all.

In [6]:
def build_models_imagetext(n_unfrozen_layers):
    assert n_unfrozen_layers >= 0, f"n_unfrozen_layers must be >= 0, got {n_unfrozen_layers}"

    dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_MODEL_NAME, cache_dir=HF_CACHE)
    dinov3_model = AutoModel.from_pretrained(DINOV3_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    for p in dinov3_model.parameters():
        p.requires_grad = False
    n_dinov3 = len(dinov3_model.layer)
    if n_unfrozen_layers > n_dinov3:
        logger.warning("n=%d > DINOv3 layers (%d); unfreezing all", n_unfrozen_layers, n_dinov3)
    if n_unfrozen_layers > 0:
        for layer in dinov3_model.layer[-n_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True
    DINOV3_DIM = dinov3_model.config.hidden_size

    def encode_dinov3_images_batched(image_paths, chunk_size=IMAGE_ENCODE_CHUNK_SIZE):
        if len(image_paths) <= chunk_size:
            images = [Image.open(p).convert("RGB") for p in image_paths]
            inputs = dinov3_processor(images=images, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            return out.last_hidden_state[:, 0, :]
        chunks = []
        for i in range(0, len(image_paths), chunk_size):
            imgs = [Image.open(p).convert("RGB") for p in image_paths[i:i + chunk_size]]
            inputs = dinov3_processor(images=imgs, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            chunks.append(out.last_hidden_state[:, 0, :])
        return torch.cat(chunks, dim=0)

    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    for p in bge_model.parameters():
        p.requires_grad = False
    n_bge = len(bge_model.encoder.layer)
    if n_unfrozen_layers > n_bge:
        logger.warning("n=%d > BGE-M3 layers (%d); unfreezing all", n_unfrozen_layers, n_bge)
    if n_unfrozen_layers > 0:
        for layer in bge_model.encoder.layer[-n_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True
    BGE_DIM = bge_model.config.hidden_size

    def encode_bge_text(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                                 max_length=BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        return out.last_hidden_state[:, 0, :]

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)
        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_bge = ProjectionHead(BGE_DIM).to(device)
    head_image_dinov3 = ProjectionHead(DINOV3_DIM).to(device)

    # --- Optimizer: only include encoder groups that have trainable params ---
    head_params      = list(head_text_bge.parameters()) + list(head_image_dinov3.parameters())
    dinov3_trainable = [p for p in dinov3_model.parameters() if p.requires_grad]
    bge_trainable    = [p for p in bge_model.parameters() if p.requires_grad]

    param_groups = [{"params": head_params, "lr": HEAD_LEARNING_RATE}]
    if dinov3_trainable:
        param_groups.append({"params": dinov3_trainable, "lr": DINOV3_ENCODER_LR})
    if bge_trainable:
        param_groups.append({"params": bge_trainable, "lr": BGE_ENCODER_LR})
    optimizer = torch.optim.AdamW(param_groups)

    logger.info("build_models_imagetext(n=%d): DINOv3 trainable params=%d, BGE trainable params=%d, "
                "optimizer groups=%d%s",
                n_unfrozen_layers, len(dinov3_trainable), len(bge_trainable), len(param_groups),
                " (frozen baseline: heads only)" if n_unfrozen_layers == 0 else "")

    return {
        "dinov3_model": dinov3_model, "encode_dinov3_images_batched": encode_dinov3_images_batched,
        "bge_model": bge_model, "encode_bge_text": encode_bge_text,
        "head_text_bge": head_text_bge, "head_image_dinov3": head_image_dinov3,
        "optimizer": optimizer,
    }

logger.info("build_models_imagetext(n) ready.")

2026-10-21 11:21:45,257 : INFO : build_models_imagetext(n) ready.


## Section 5 — FN-masked contrastive loss (verbatim from NB28)

`masked_symmetric_infonce` + `compute_imagetext_loss` — same math as NB28. The diagonal (positives) is never masked; FN pairs are masked out of the softmax.

In [7]:
def make_batches(records, batch_size=BATCH_SIZE, shuffle=True):
    idx = np.arange(len(records))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield [records[i] for i in batch_idx]


def masked_symmetric_infonce(sim_matrix, mask_matrix, temperature=TEMPERATURE):
    B = sim_matrix.shape[0]
    logits = sim_matrix / temperature
    neg_inf = torch.finfo(logits.dtype).min
    mask_matrix = mask_matrix.clone()
    mask_matrix.fill_diagonal_(False)
    logits = logits.masked_fill(mask_matrix, neg_inf)
    targets = torch.arange(B, device=logits.device)
    loss_i2j = F.cross_entropy(logits, targets)
    loss_j2i = F.cross_entropy(logits.T, targets)
    return (loss_i2j + loss_j2i) / 2


def compute_imagetext_loss(batch, models):
    texts = [r["text"] for r in batch]
    image_paths = [r["image_path"] for r in batch]
    B = len(batch)
    text_emb_raw = models["encode_bge_text"](texts)
    image_embs_raw = models["encode_dinov3_images_batched"](image_paths)
    text_emb = models["head_text_bge"](text_emb_raw)
    image_emb = models["head_image_dinov3"](image_embs_raw)
    sim = text_emb @ image_emb.T
    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        for j in range(B):
            if i != j and is_masked_image_image(batch[i], batch[j]):
                mask[i, j] = True
    return masked_symmetric_infonce(sim, mask)

logger.info("compute_imagetext_loss() (FN-masked) ready.")

2026-10-21 11:21:45,393 : INFO : compute_imagetext_loss() (FN-masked) ready.


## Section 6 — GT metrics + evaluation function (aligned with NB28)

Image encoding batched + chunked via `encode_dinov3_images_batched`. BGE text at `BGE_MAX_LENGTH=8192` — **text chunked via `encode_bge_text_chunked_eval`**, required at that max-len (unchunked 500×8192 OOM'd in NB28).

In [8]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def encode_bge_text_chunked_eval(texts, models, chunk_size=GT_EVAL_CHUNK_SIZE):
    if len(texts) <= chunk_size:
        return models["encode_bge_text"](texts)
    chunks = []
    for i in range(0, len(texts), chunk_size):
        chunks.append(models["encode_bge_text"](texts[i:i + chunk_size]))
    return torch.cat(chunks, dim=0)


def evaluate_imagetext(models):
    models["head_text_bge"].eval(); models["head_image_dinov3"].eval()
    models["dinov3_model"].eval(); models["bge_model"].eval()
    with torch.no_grad():
        image_embs_raw = models["encode_dinov3_images_batched"](
            [r["image_path"] for r in gt_textimage_records]
        )
        gt_image_emb = models["head_image_dinov3"](image_embs_raw)
        gt_text_emb = models["head_text_bge"](
            encode_bge_text_chunked_eval([r["text"] for r in gt_textimage_records], models)
        )
    sim = gt_image_emb @ gt_text_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]
    models["head_text_bge"].train(); models["head_image_dinov3"].train()
    models["dinov3_model"].train(); models["bge_model"].train()
    return summary

logger.info("evaluate_imagetext() ready.")

2026-10-21 11:21:45,459 : INFO : evaluate_imagetext() ready.


## Section 7 — Sanity check (run BEFORE the sweep)

One full loss + backward on `n = N_LAYERS_TO_TRY[0]` — **`n = 0`** by default. Exercises the frozen-encoder path so head-only baseline problems surface immediately.

**Gradient check is `n`-aware:**
- `n = 0` — absence of encoder gradients is **correct**; logs "expected" and asserts heads *did* receive gradients.
- `n > 0` — absence of encoder gradients is a **bug**; **raises**.

In [9]:
_sanity_n = N_LAYERS_TO_TRY[0]
_sanity_models = build_models_imagetext(_sanity_n)
_sanity_batch = textimage_records[:BATCH_SIZE]
_loss_check = compute_imagetext_loss(_sanity_batch, _sanity_models)

logger.info("Sanity (n=%d): loss=%.4f (naive ln(%d)=%.4f)",
            _sanity_n, _loss_check.item(), BATCH_SIZE, math.log(BATCH_SIZE))
if not math.isfinite(_loss_check.item()):
    raise RuntimeError("Sanity loss is NaN/Inf -- stop and debug before running the sweep")

_loss_check.backward()

_grad_dinov3 = any(p.grad is not None and p.grad.abs().sum().item() > 0
                   for p in _sanity_models["dinov3_model"].parameters() if p.requires_grad)
_grad_bge = any(p.grad is not None and p.grad.abs().sum().item() > 0
                for p in _sanity_models["bge_model"].parameters() if p.requires_grad)

if _sanity_n == 0:
    logger.info("Sanity (n=0): both encoders FROZEN -- absence of encoder gradients is expected.")
    logger.info("  dinov3 trainable params receive grad: %s (expected False)", _grad_dinov3)
    logger.info("  bge    trainable params receive grad: %s (expected False)", _grad_bge)
    _grad_head = any(p.grad is not None and p.grad.abs().sum().item() > 0
                     for p in _sanity_models["head_text_bge"].parameters()) or \
                 any(p.grad is not None and p.grad.abs().sum().item() > 0
                     for p in _sanity_models["head_image_dinov3"].parameters())
    logger.info("  projection heads receive grad: %s (expected True)", _grad_head)
    if not _grad_head:
        raise RuntimeError("Sanity failed at n=0: no gradients reaching projection heads either")
else:
    logger.info("Gradients reaching unfrozen DINOv3 layers: %s", _grad_dinov3)
    logger.info("Gradients reaching unfrozen BGE layers: %s", _grad_bge)
    if not (_grad_dinov3 and _grad_bge):
        raise RuntimeError(
            f"Sanity failed at n={_sanity_n}: encoders should be receiving gradients "
            f"(dinov3={_grad_dinov3}, bge={_grad_bge}). Check optimizer param groups."
        )

del _sanity_models, _loss_check, _sanity_batch
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache(); torch.cuda.synchronize()
logger.info("Allocated after sanity cleanup: %.2f GB", torch.cuda.memory_allocated() / 1e9)

2026-10-21 11:21:50,343 : INFO : build_models_imagetext(n=0): DINOv3 trainable params=0, BGE trainable params=0, optimizer groups=1 (frozen baseline: heads only)
2026-10-21 11:21:55,222 : INFO : Sanity (n=0): loss=3.5603 (naive ln(32)=3.4657)
2026-10-21 11:21:55,261 : INFO : Sanity (n=0): both encoders FROZEN -- absence of encoder gradients is expected.
2026-10-21 11:21:55,263 : INFO :   dinov3 trainable params receive grad: False (expected False)
2026-10-21 11:21:55,265 : INFO :   bge    trainable params receive grad: False (expected False)
2026-10-21 11:21:55,273 : INFO :   projection heads receive grad: True (expected True)
2026-10-21 11:21:55,960 : INFO : Allocated after sanity cleanup: 0.02 GB


## Section 8 — Train-or-resume for one `n` (GT every 2 epochs)

Layout under `OUTPUT_PARENT/last_{n}_layers/` matches NB28 / NB29's `size_{N}/` — same filenames, same checkpoint structure.

**GT cadence:** inside the epoch loop, evaluate + save a `per_epoch_gt/epoch_NN_gt_results.csv` **only when `(epoch+1) % GT_EVERY_N_EPOCHS == 0`, or on the last epoch** (`target_epochs`) — so 10 CSVs per run at `[2,4,6,...,20]`. `training_history.csv` has the same columns as NB28 (`epoch`, `imagetext_loss`).

**Controlled batch order:** `np.random.seed(RANDOM_SEED)` before the epoch loop ensures each `n` sees the identical batch sequence.

In [10]:
def train_or_resume_for_n_layers(n_unfrozen_layers, output_parent, size, records_pool, batch_size,
                                  build_models_fn, loss_fn, eval_fn, state_dict_keys, loss_key,
                                  target_epochs=TARGET_EPOCHS, gt_every_n_epochs=GT_EVERY_N_EPOCHS):
    size_dir = os.path.join(output_parent, f"last_{n_unfrozen_layers}_layers")
    os.makedirs(size_dir, exist_ok=True)
    ckpt_path = os.path.join(size_dir, "latest_checkpoint.pt")
    gt_results_path = os.path.join(size_dir, "gt_results.csv")
    per_epoch_gt_dir = os.path.join(size_dir, "per_epoch_gt")

    label = f"L{n_unfrozen_layers}"

    # Case 1: already complete
    if os.path.exists(gt_results_path):
        logger.info("[%s] Already completed (%s exists) -- skipping.", label, gt_results_path)
        try:
            return pd.read_csv(gt_results_path)
        except Exception as e:
            logger.warning("[%s] Failed to read existing %s: %s", label, gt_results_path, e)
            return None

    os.makedirs(per_epoch_gt_dir, exist_ok=True)

    # SAME 2500 subset for every n (seed depends on SIZE, not n)
    rng = random.Random(RANDOM_SEED + size)
    size_records = list(records_pool) if size >= len(records_pool) else rng.sample(records_pool, size)
    logger.info("[%s] Records: %d (fixed subset, seed=%d)", label, len(size_records), RANDOM_SEED + size)

    models = build_models_fn(n_unfrozen_layers)

    # Case 2/3 (ckpt present) or Case 4 (missing -> fresh)
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        start_epoch = ckpt["epoch"]
        for key in state_dict_keys:
            models[key].load_state_dict(ckpt[key])
        models["optimizer"].load_state_dict(ckpt["optimizer"])
        global_step = ckpt["global_step"]
        history = ckpt.get("history", {"epoch": []})
        if start_epoch >= target_epochs:
            logger.info("=" * 70)
            logger.info("=== %s: checkpoint at target (%d/%d) -- finalizing without training ===",
                        label, start_epoch, target_epochs)
            logger.info("=" * 70)
        else:
            logger.info("=" * 70)
            logger.info("=== %s: resuming epoch %d -> target %d ===", label, start_epoch, target_epochs)
            logger.info("=" * 70)
        logger.info("[%s] Loaded: epoch %d, step %d, history keys=%s",
                    label, start_epoch, global_step, list(history.keys()))
    else:
        start_epoch = 0
        global_step = 0
        history = {"epoch": []}
        logger.info("=" * 70)
        logger.info("=== %s: NO checkpoint at %s -- training FRESH from epoch 0 -> target %d ===",
                    label, ckpt_path, target_epochs)
        logger.info("=" * 70)

    def save_checkpoint(epoch, global_step, history):
        tmp_path = ckpt_path + ".tmp"
        state = {key: models[key].state_dict() for key in state_dict_keys}
        state["optimizer"] = models["optimizer"].state_dict()
        state["epoch"] = epoch
        state["global_step"] = global_step
        state["history"] = history
        torch.save(state, tmp_path)
        os.replace(tmp_path, ckpt_path)

    # Controlled comparison: identical shuffle sequence across all n
    np.random.seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)

    training_start_time = time.time()
    stop_early = False

    try:
        for epoch in range(start_epoch, target_epochs):
            epoch_losses = []
            batches = list(make_batches(size_records, batch_size))
            n_steps = len(batches)

            for step in range(n_steps):
                models["optimizer"].zero_grad()
                loss = loss_fn(batches[step], models)
                loss.backward()
                models["optimizer"].step()
                global_step += 1
                if device == "cuda":
                    torch.cuda.empty_cache()
                epoch_losses.append(loss.item())

                elapsed_hours = (time.time() - training_start_time) / 3600
                if step % LOG_EVERY_N_STEPS == 0:
                    logger.info("[%s] Epoch %d/%d, step %d/%d, loss=%.4f, elapsed=%.2fh",
                                label, epoch + 1, target_epochs, step, n_steps, loss.item(), elapsed_hours)

                if global_step % CHECKPOINT_EVERY_N_STEPS == 0:
                    save_checkpoint(epoch, global_step, history)

                if elapsed_hours > MAX_RUNTIME_HOURS_PER_SIZE:
                    logger.warning("[%s] Runtime limit reached. Saving and stopping.", label)
                    save_checkpoint(epoch, global_step, history)
                    stop_early = True
                    break
            if stop_early:
                break

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
            history["epoch"].append(epoch + 1)
            history.setdefault(loss_key, []).append(mean_loss)
            logger.info("[%s] Epoch %d done: loss=%.4f", label, epoch + 1, mean_loss)
            save_checkpoint(epoch + 1, global_step, history)

            # --- GT every GT_EVERY_N_EPOCHS, plus the last epoch always ---
            is_gt_epoch = ((epoch + 1) % gt_every_n_epochs == 0) or ((epoch + 1) == target_epochs)
            if is_gt_epoch:
                epoch_gt_summary = eval_fn(models)
                epoch_gt_summary.insert(0, "train_size", size)
                epoch_gt_summary.insert(1, "epoch", epoch + 1)
                epoch_gt_summary.insert(2, "n_unfrozen_layers", n_unfrozen_layers)
                epoch_gt_path = os.path.join(per_epoch_gt_dir, f"epoch_{epoch + 1:02d}_gt_results.csv")
                epoch_gt_summary.to_csv(epoch_gt_path, index=False)
                logger.info("[%s] Epoch %d GT results:\n%s",
                            label, epoch + 1,
                            epoch_gt_summary.drop(columns=["train_size", "n_unfrozen_layers"]).to_string(index=False))
            else:
                logger.info("[%s] Epoch %d: GT skipped (cadence=%d)", label, epoch + 1, gt_every_n_epochs)

        # --- Final artefacts (filenames match NB28) ---
        pd.DataFrame(history).to_csv(os.path.join(size_dir, "training_history.csv"), index=False)
        for key in state_dict_keys:
            fname_stem = _MODEL_FILE_RENAMES.get(key, key)
            torch.save(models[key].state_dict(), os.path.join(size_dir, f"{fname_stem}.pt"))
        logger.info("[%s] Saved updated outputs.", label)

        gt_summary = eval_fn(models)
        gt_summary.insert(0, "train_size", size)
        gt_summary.insert(1, "n_unfrozen_layers", n_unfrozen_layers)
        gt_summary.to_csv(gt_results_path, index=False)
        logger.info("[%s] Final GT results:\n%s", label, gt_summary.to_string(index=False))
        return gt_summary

    except torch.cuda.OutOfMemoryError:
        logger.error("[%s] OOM. Skipping.", label)
        return None
    finally:
        try:
            del models
        except NameError:
            pass
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache(); torch.cuda.synchronize()

logger.info("train_or_resume_for_n_layers() ready.")

2026-10-21 11:21:55,979 : INFO : train_or_resume_for_n_layers() ready.


## Section 9 — Sweep over `n`

`n = 0` runs first — if the head-only baseline itself reveals a bug, you find out before spending the sweep's compute on `n=2,4,8`. `n=6` is not in the sweep (NB28 reference — see top).

In [11]:
results = []
for n in N_LAYERS_TO_TRY:
    logger.info("#" * 70)
    logger.info("### SWEEP: n_unfrozen_layers = %d %s###",
                n, "(frozen baseline) " if n == 0 else "")
    logger.info("#" * 70)
    r = train_or_resume_for_n_layers(
        n_unfrozen_layers=n, output_parent=OUTPUT_PARENT, size=SIZE, records_pool=textimage_records,
        batch_size=BATCH_SIZE, build_models_fn=build_models_imagetext,
        loss_fn=compute_imagetext_loss, eval_fn=evaluate_imagetext,
        state_dict_keys=["head_text_bge", "head_image_dinov3", "dinov3_model", "bge_model"],
        loss_key="imagetext_loss",
    )
    if r is not None:
        results.append(r)

logger.info("Sweep complete. %d n values produced results.", len(results))

2026-10-21 11:21:56,140 : INFO : ######################################################################
2026-10-21 11:21:56,142 : INFO : ### SWEEP: n_unfrozen_layers = 0 (frozen baseline) ###
2026-10-21 11:21:56,144 : INFO : ######################################################################
2026-10-21 11:21:56,146 : INFO : [L0] Already completed (/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_fn_masked/last_0_layers/gt_results.csv exists) -- skipping.
2026-10-21 11:21:56,157 : INFO : ######################################################################
2026-10-21 11:21:56,158 : INFO : ### SWEEP: n_unfrozen_layers = 2 ###
2026-10-21 11:21:56,160 : INFO : ######################################################################
2026-10-21 11:21:56,162 : INFO : [L2] Already completed (/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_fn_masked/last_2_layers/gt_results.csv exists) -- skipping.
2026-10-21 11:21:56,167 : INFO : ###############

## Section 10 — Combined summaries

Two roll-ups under `OUTPUT_PARENT/`, aligned with NB28 / NB29's `summary_all_sizes.csv`:

- **`summary_all_n.csv`** — final GT per `n`. Columns: `train_size, n_unfrozen_layers, source, direction, N, R@1, R@3, R@5, R@10, MRR`.
  - Rows from NB27's sweep are tagged `source=NB27`.
  - If NB28's `size_2500/gt_results.csv` exists, it is loaded as `n=6, source=NB28_ref`. Otherwise a warning is logged and the `n=6` row is simply absent.
- **`summary_per_epoch_all_n.csv`** — every `(n, epoch, direction)` row from NB27's `per_epoch_gt/` (epochs 2, 4, 6, …, 20). `n=6` is **not** present here (NB28 does not save per-epoch CSVs).

In [12]:
def load_nb28_reference(gt_path, expected_size=SIZE, reference_n=6):
    """Load NB28's size_2500 gt_results.csv as the n=6 reference.
    NB28 uses FN masking at n=6 -- identical objective to NB27 n=6.
    NB29 (in-batch mining) is NOT a valid reference and is not consulted here."""
    if not os.path.exists(gt_path):
        logger.warning("[REF n=%d] NB28 reference not found at %s", reference_n, gt_path)
        logger.warning("           summary_all_n.csv will lack the n=%d reference row.", reference_n)
        logger.warning("           Run NB28 for size %d (add %d to IMG_SIZES_TO_RESUME), then re-run this cell.",
                       expected_size, expected_size)
        return None
    ref = pd.read_csv(gt_path)
    if "train_size" in ref.columns and not (ref["train_size"] == expected_size).all():
        logger.warning("[REF n=%d] NB28 file %s has train_size=%s, expected %d -- skipping",
                       reference_n, gt_path, ref["train_size"].unique().tolist(), expected_size)
        return None
    ref = ref.copy()
    if "train_size" not in ref.columns:
        ref.insert(0, "train_size", expected_size)
    ref.insert(1, "n_unfrozen_layers", reference_n)
    ref.insert(2, "source", "NB28_ref")
    logger.info("[REF n=%d] Loaded NB28 reference from %s (%d rows)", reference_n, gt_path, len(ref))
    return ref


def collect_summary_final(output_parent, nb28_ref_path=None):
    """Final GT per n. Includes NB28's n=6 reference if its file exists."""
    rows = []
    for entry in sorted(os.listdir(output_parent)):
        m = re.match(r"last_(\d+)_layers$", entry)
        if not m:
            continue
        p = os.path.join(output_parent, entry, "gt_results.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["source"] = "NB27"
            rows.append(df)
        else:
            logger.warning("No gt_results.csv at %s", p)

    if nb28_ref_path is not None:
        ref = load_nb28_reference(nb28_ref_path, expected_size=SIZE, reference_n=6)
        if ref is not None:
            rows.append(ref)

    if not rows:
        return None
    df = pd.concat(rows, ignore_index=True)
    first_cols = ["train_size", "n_unfrozen_layers", "source", "direction",
                  "N", "R@1", "R@3", "R@5", "R@10", "MRR"]
    df = df[[c for c in first_cols if c in df.columns]]
    df = (df.sort_values(["n_unfrozen_layers", "source", "direction"])
            .reset_index(drop=True))
    out = os.path.join(output_parent, "summary_all_n.csv")
    df.to_csv(out, index=False)
    logger.info("Final summary written to %s:\n%s", out, df.to_string(index=False))
    return df


def collect_summary_per_epoch(output_parent):
    """Per-(n, epoch, direction) rows. Only NB27's own runs."""
    rows = []
    for entry in sorted(os.listdir(output_parent)):
        m = re.match(r"last_(\d+)_layers$", entry)
        if not m:
            continue
        per_epoch_dir = os.path.join(output_parent, entry, "per_epoch_gt")
        if not os.path.isdir(per_epoch_dir):
            continue
        for f in sorted(os.listdir(per_epoch_dir)):
            if f.startswith("epoch_") and f.endswith("_gt_results.csv"):
                rows.append(pd.read_csv(os.path.join(per_epoch_dir, f)))
    if not rows:
        return None
    df = (pd.concat(rows, ignore_index=True)
            .drop_duplicates(subset=["n_unfrozen_layers", "epoch", "direction"], keep="last")
            .sort_values(["n_unfrozen_layers", "epoch", "direction"])
            .reset_index(drop=True))
    out = os.path.join(output_parent, "summary_per_epoch_all_n.csv")
    df.to_csv(out, index=False)
    logger.info("Per-epoch summary written to %s (%d rows)", out, len(df))
    return df


summary_all_n = collect_summary_final(OUTPUT_PARENT, nb28_ref_path=NB28_N6_REFERENCE_GT_PATH)
summary_per_epoch_all_n = collect_summary_per_epoch(OUTPUT_PARENT)

try:
    display(summary_all_n)
except NameError:
    print(summary_all_n)

if summary_per_epoch_all_n is not None:
    last_epoch = summary_per_epoch_all_n["epoch"].max()
    compact = summary_per_epoch_all_n[summary_per_epoch_all_n["epoch"] == last_epoch]
    try:
        display(compact)
    except NameError:
        print(compact)

2026-10-21 11:22:11,510 : INFO : [REF n=6] Loaded NB28 reference from /projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/Smaller_train_sizes_runs/size_2500/gt_results.csv (2 rows)
2026-10-21 11:22:11,624 : INFO : Final summary written to /projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_fn_masked/summary_all_n.csv:
 train_size  n_unfrozen_layers   source     direction   N   R@1   R@3   R@5  R@10      MRR
       2500                  0     NB27 image -> text 500 0.196 0.346 0.408 0.530 0.304546
       2500                  0     NB27 text -> image 500 0.202 0.338 0.408 0.518 0.306399
       2500                  2     NB27 image -> text 500 0.194 0.332 0.428 0.542 0.302794
       2500                  2     NB27 text -> image 500 0.188 0.340 0.424 0.520 0.300945
       2500                  4     NB27 image -> text 500 0.218 0.378 0.438 0.516 0.325802
       2500                  4     NB27 text -> image 500 0.220 0.374 0.428 0.526 0.327458
       2500   

,train_size,n_unfrozen_layers,source,direction,N,R@1,R@3,R@5,R@10,MRR
0,2500,0,NB27,image -> text,500,0.196,0.346,0.408,0.530,0.304546
1,2500,0,NB27,text -> image,500,0.202,0.338,0.408,0.518,0.306399
2,2500,2,NB27,image -> text,500,0.194,0.332,0.428,0.542,0.302794
3,2500,2,NB27,text -> image,500,0.188,0.340,0.424,0.520,0.300945
4,2500,4,NB27,image -> text,500,0.218,0.378,0.438,0.516,0.325802
5,2500,4,NB27,text -> image,500,0.220,0.374,0.428,0.526,0.327458
6,2500,6,NB27,image -> text,500,0.214,0.370,0.424,0.546,0.323403
7,2500,6,NB27,text -> image,500,0.214,0.366,0.462,0.560,0.329059
8,2500,6,NB28_ref,image -> text,500,0.198,0.364,0.426,0.534,0.313464
9,2500,6,NB28_ref,text -> image,500,0.216,0.354,0.440,0.530,0.325603


,train_size,epoch,n_unfrozen_layers,direction,N,R@1,R@3,R@5,R@10,MRR
18,2500,20,0,image -> text,500,0.196,0.346,0.408,0.530,0.304546
19,2500,20,0,text -> image,500,0.202,0.338,0.408,0.518,0.306399
38,2500,20,2,image -> text,500,0.194,0.332,0.428,0.542,0.302794
39,2500,20,2,text -> image,500,0.188,0.340,0.424,0.520,0.300945
58,2500,20,4,image -> text,500,0.218,0.378,0.438,0.516,0.325802
59,2500,20,4,text -> image,500,0.220,0.374,0.428,0.526,0.327458
78,2500,20,6,image -> text,500,0.214,0.370,0.424,0.546,0.323403
79,2500,20,6,text -> image,500,0.214,0.366,0.462,0.560,0.329059
98,2500,20,8,image -> text,500,0.208,0.352,0.418,0.552,0.316096
99,2500,20,8,text -> image,500,0.206,0.366,0.432,0.554,0.322012
